# Lab 15: Optimisation with TensorRT

**Duration:** about 2 hours. **Level:** Labs 1 to 14 completed.

| Student name | Station no. | Date |
|---|---|---|
| | | |

(Double-click this cell to type your name.)

## 1. Overview
In Lab 10 you ran AI models on the GPU. They worked, but they were not as fast as the Jetson can go. In this lab you optimise them. You convert a model to TensorRT, NVIDIA's tool for making a model run faster on its own GPU, and you measure exactly how much faster it becomes.

The FP16 engine you build here is used in Lab 16, the mini project.

**By the end of this lab you will be able to:**
- Explain what TensorRT does and why an engine file belongs to one kind of Jetson.
- Explain the difference between FP32 and FP16 numbers.
- Export a YOLO model to TensorRT engines in FP32 and FP16.
- Benchmark a model before and after optimisation, and calculate the speed-up.
- Check that an optimised model still gives the same detections.
- Measure how the power mode changes the speed.
- Choose a power mode to match the needs of a job.

**Prerequisites:** Labs 1 to 14 completed, with your Lab 10 benchmark table (`yolo26n.pt`). A USB camera chosen for you by your teacher. PyTorch, Ultralytics and the models were installed in Lab 0, and TensorRT comes with JetPack 6.2. You install nothing in this lab.

> **Memory.** Building an engine uses most of the shared 8 GB. Before Part B, close your other notebooks: choose **Kernel, Shut Down All Kernels** in the menu, then reopen only this notebook. If several students share the Jetson, the builds run **one after the other**: the build cell tells you when you wait for a classmate.

## 2. Background

### 2.1 Why Optimise?
A model that runs at 15 frames per second may be too slow to follow a moving object. A model that runs at 60 frames per second can also use less power for the same job, because the GPU finishes sooner. On a small board with limited memory and a battery, faster and lighter models decide what is possible.

### 2.2 TensorRT
TensorRT takes a trained model and builds an engine for one specific GPU. While building, it tries different ways to run each part, joins small steps into one (called layer fusion), and chooses the fastest. The build takes minutes, but it is done only once. After that every run uses the fast engine.

| Term | Meaning |
|---|---|
| Engine (`.engine`) | The optimised model file made by TensorRT for this GPU and this TensorRT version |
| Build (export) | Making the engine from the `.pt` file. Slow, and done once. |
| Run (inference) | Using the engine on pictures. Fast, and done many times. |
| Speed-up | Old time divided by new time. A speed-up of 3 means three times faster. |

> **Warning.** An engine only works on the same type of GPU and the same TensorRT and CUDA versions that built it. If you copy an engine to another kind of device, or after a JetPack update, it can fail or give wrong results. Build it again on the target device. This is also why every student builds their own engines in their own folder.

### 2.3 FP32 and FP16
A model stores its numbers with a certain precision. FP32 uses 32 bits for each number. FP16 uses 16 bits, which is half the memory, and the GPU can calculate with them much faster. The numbers are slightly less exact, but for most vision models the difference in the answers is tiny.

| Precision | Bits per number | Speed | Accuracy |
|---|---|---|---|
| FP32 | 32 | Slowest | Highest |
| FP16 | 16 | Faster, less memory | Almost the same |
| INT8 | 8 | Fastest | Can drop. Needs extra steps. |

This lab uses FP32 and FP16. INT8 needs a calibration step with sample pictures and is not covered here.

> **Note.** Always check the result after optimising, not only the speed. A faster model that finds fewer objects is not an improvement. You do this check in Part C.

### 2.4 Power Modes Again
In Lab 1 you saw that the power mode limits the clocks of the CPU and GPU. A model runs faster in MAXN SUPER than in 15W, but the board uses more power and runs hotter. The right mode depends on the job. You measure this in Part E.

## 3. Procedure
Every file this lab makes (`.onnx`, `.engine` and pictures) is saved in the folder of this notebook, which is `lab15` inside your own folder.

## Part A: Start Point and Checks (10 min)

### Step 1: Create the Workspace and Allow `sudo`
Copy the model and the test picture from your `models` folder (filled in Lab 0). Then run the `sudo` cell once and type your password.

In [ ]:
%%bash
pwd
cp -n "$LAB_HOME"/models/yolo26n.pt "$LAB_HOME"/models/bus.jpg .
ls

In [ ]:
import labview
labview.sudo_login()

### Step 2: Check TensorRT
This writes a small check program into the folder (`%%writefile` shows the program and saves it), then runs it with the `python` command.

In [ ]:
%%writefile check_trt.py
import torch
import tensorrt
import ultralytics

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("TensorRT version:", tensorrt.__version__)
print("Ultralytics version:", ultralytics.__version__)

In [ ]:
!python check_trt.py

| Item | Your result |
|---|---|
| PyTorch CUDA available | |
| TensorRT version | |
| Ultralytics version | |

### Step 3: Choose the Power Mode for Parts A to D
Use the same power mode as in Lab 10, so the comparison is fair. Check it and copy your Lab 10 result from your benchmark table:

In [ ]:
!sudo nvpmodel -q

| Item | Your result |
|---|---|
| Power mode now | |
| Lab 10 power mode | |
| Lab 10 result for `yolo26n.pt` (ms and FPS) | |

If the mode is not MAXN SUPER, switch with `!sudo nvpmodel -m 2`.

## Part B: Build TensorRT Engines (30 min)

### Step 1: Export FP32 and FP16 Engines
This program exports `yolo26n.pt` twice: once at the default 32-bit precision and once at 16-bit. Each engine is saved under its own name. It tries `quantize=16` first and, if your version of Ultralytics does not know it, it tries `half=True`.

In [ ]:
%%writefile export_trt.py
import os
import time
from ultralytics import YOLO

name = "yolo26n"


def build(label, **precision):
    model = YOLO(f"{name}.pt")
    start = time.time()
    path = model.export(format="engine", device=0, **precision)
    new_name = f"{name}_{label}.engine"
    os.replace(path, new_name)   # give each engine its own file name
    size = os.path.getsize(new_name) / 1e6
    print(f"Built {new_name} in {time.time() - start:.0f} s, size {size:.1f} MB")


build("fp32")                    # default precision: 32-bit

try:
    build("fp16", quantize=16)   # 16-bit
except Exception as error:       # older Ultralytics versions use half=True
    print("quantize=16 did not work:", error)
    print("Trying half=True instead")
    build("fp16", half=True)

Run the build. It takes about **12 minutes** (about 4 for FP32 and 8 for FP16). `build_lock()` makes students take turns, because one build needs most of the memory. The output shows many lines and may look stuck for a while.

> **Warning.** Do not press the stop button and do not unplug anything. Wait until you see `Built yolo26n_fp16.engine`. Red or yellow lines that start with `[TRT]` are normal.

In [ ]:
import labview
with labview.build_lock():
    !python export_trt.py

In [ ]:
!ls -lh *.engine *.onnx *.pt

| File | Size | Built in (s) | Precision |
|---|---|---|---|
| `yolo26n.pt` | | - | FP32 |
| `yolo26n.onnx` | | - | FP32 |
| `yolo26n_fp32.engine` | | | FP32 |
| `yolo26n_fp16.engine` | | | FP16 |

> **Note.** The `.onnx` file is a middle step. TensorRT reads the ONNX file and builds the engine from it. If the export stops with an error, call your instructor before you try again.

## Part C: Benchmark and Check the Result (20 min)

### Step 1: Compare Speed
This program runs the original model and both engines on `bus.jpg`. For each one it prints the average time over 30 runs after a warm-up, the size of the file, and the number of objects found.

In [ ]:
%%writefile benchmark_trt.py
import os
import sys
import torch
from ultralytics import YOLO

image = sys.argv[1] if len(sys.argv) > 1 else "bus.jpg"
device = 0 if torch.cuda.is_available() else "cpu"
files = ["yolo26n.pt", "yolo26n_fp32.engine", "yolo26n_fp16.engine"]
runs = 30

for f in files:
    if not os.path.exists(f):
        print(f"{f}: file not found, skipped")
        continue

    model = YOLO(f, task="detect")
    for _ in range(5):                          # warm-up
        model(image, device=device, verbose=False)

    times = []
    for _ in range(runs):
        r = model(image, device=device, verbose=False)[0]
        times.append(r.speed["inference"])      # milliseconds, model only

    avg = sum(times) / len(times)
    size = os.path.getsize(f) / 1e6
    print(f"{f:<22} {avg:6.1f} ms  {1000 / avg:6.1f} FPS  "
          f"{size:5.1f} MB  objects found: {len(r.boxes)}")

Run it **twice** and record the second run. Calculate the speed-up for each engine as old time / new time, using the `.pt` time as the old time.

In [ ]:
!python benchmark_trt.py

| Model file | Time (ms) | FPS | Size (MB) | Objects | Speed-up |
|---|---|---|---|---|---|
| `yolo26n.pt` | | | | | 1.0 |
| `yolo26n_fp32.engine` | | | | | |
| `yolo26n_fp16.engine` | | | | | |

**Reading the output**
- Lines that start with `[TRT] [I]` come from TensorRT. They only say that the engine was loaded. A warning from `onnxruntime` about `device_discovery` is also normal. You can ignore both.
- Time is the average for one picture in milliseconds. A smaller number is faster. FPS is 1000 / time.
- Speed-up is the `.pt` time divided by the engine time. For example 29 ms divided by 13 ms is about 2.2, so the engine is more than twice as fast.
- Objects should stay the same for all three files. Here that is 5 objects in `bus.jpg`.
- The FP32 engine is bigger than the `.pt` file, and the FP16 engine is smaller than the FP32 engine.
- The `.pt` time can change by several ms from one run to the next, because plain PyTorch spends part of its time on the CPU. The engine times stay almost the same. Compare the engines with each other, and use the `.pt` time only as a rough guide.

### Step 2: Is the Result Still Correct?
Look at the Objects column. All three models should find about the same number of objects in the same picture. A change of one low-confidence object is normal for FP16. A big change is not.

| Question | Your answer |
|---|---|
| Do all three find the same number of objects? | |
| Which model file is the smallest, and which is the fastest? | |
| Is the Lab 10 time for `yolo26n.pt` similar to today's? | |

## Part D: Live FPS With TensorRT (15 min)
The live picture appears inside the notebook. The FPS is written on the picture. Run it twice, one after the other: first with the plain model, then with the FP16 engine. Press **Stop** after about 20 seconds each, and keep the same objects in view for both runs.

In [ ]:
import cv2, torch
from ultralytics import YOLO
import labview

device = 0 if torch.cuda.is_available() else "cpu"

def start_live(name):
    model = YOLO(name, task="detect")
    print("Using model:", name)

    def process(frame, state):
        result = model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
        shown = result.plot()
        counts = {}
        for c in result.boxes.cls:
            kind = result.names[int(c)]
            counts[kind] = counts.get(kind, 0) + 1
        summary = ", ".join(f"{n}: {k}" for n, k in counts.items()) or "nothing found"
        cv2.putText(shown, f"{state['fps']:.0f} FPS  conf {state['Confidence']:.2f}", (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(shown, summary, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        return shown

    return labview.live(process, {"Confidence": (0.05, 0.95, 0.05, 0.40)})

**Run 1:** the plain PyTorch model. Press **Stop** when you have read the FPS.

In [ ]:
view = start_live("yolo26n.pt")

**Run 2:** the FP16 TensorRT engine. Press **Stop** when you have read the FPS.

In [ ]:
view = start_live("yolo26n_fp16.engine")

| Model file | FPS shown | Looks smoother? |
|---|---|---|
| `yolo26n.pt` | | |
| `yolo26n_fp16.engine` | | |

> **Note.** In a notebook the picture is also compressed and sent to your laptop, so the FPS is lower than on the Jetson's own screen. If both runs show almost the same FPS, the camera or the browser is the limit (often 30 FPS). The engine is still faster. The gain just does not show on the screen. Part C shows the real model speed.

## Part E: Speed and Power Modes (15 min)
Now measure how much the power mode matters. First look up the mode numbers, as in Lab 1:

In [ ]:
!grep POWER_MODEL /etc/nvpmodel.conf

For each mode, the next cell switches to it, confirms it, waits 10 seconds for the clocks to settle, and runs the benchmark. It prints only the lines with the speed, and it ends in MAXN SUPER. It needs about 2 minutes. Open jtop in a **Terminal tab** (File, New, Terminal) and note the GPU temperature right after each run.

In [ ]:
%%bash
for m in 0 1 2; do
  echo "==== mode $m"
  sudo nvpmodel -m $m
  sudo nvpmodel -q | head -n 1
  sleep 10
  python benchmark_trt.py 2>&1 | grep -E 'FPS'
done

| Power mode | Mode no. | `.pt` time (ms) | FP16 time (ms) | GPU temp (C) |
|---|---|---|---|---|
| 15W | | | | |
| 25W | | | | |
| MAXN SUPER | 2 | | | |

The `.pt` time is not steady (see Part C), so it may not follow the power mode. The engine times are steady.

Which model reacts more to the power mode, the `.pt` model or the FP16 engine? Finally, return to the power mode you want for the mini project. Write your answer, your choice and the reason (double-click this cell):

> **Safety.** The heatsink can be hot after running in MAXN SUPER. Do not touch it. Make sure the fan is running and the air vents are clear.

## 4. Check Task
Show the following to your instructor. The instructor initials each line when it is done correctly.

1. `check_trt.py` shows CUDA available and a TensorRT version.
2. Both engine files exist, with sizes, and you explain why the export took so long.
3. `benchmark_trt.py` shows the three models and the speed-up table is complete.
4. You explain whether the results stayed the same (the Objects column) and why this matters.
5. The live view runs the FP16 engine and the FPS table in Part D is complete.
6. The power mode table in Part E is complete and you explain the speed and heat trade-off.
7. You can say which power mode you would choose for the mini project and why.
8. The tables in Parts A to E are complete.

## 5. Assessment Questions
Answer in your own words in this cell. One or two sentences are enough unless the question says otherwise.

1. What does TensorRT do to a model? Why is the export slow but the later runs fast?
2. Why must an engine be built again after a JetPack update or on a different type of GPU?
3. What is the difference between FP32 and FP16? Give one benefit and one risk of using FP16.
4. The original model takes 16 ms per picture and the FP16 engine takes 5 ms. Calculate the speed-up and the FPS of each.
5. After optimising, the engine is much faster but finds fewer objects in the test picture. Is this a good result? What should you do?
6. Why does the same engine run faster in MAXN SUPER than in 15W? Give one cost of using MAXN SUPER.
7. Why does the benchmark run five warm-up passes, and why does it average 30 runs instead of using one?
8. You copy an engine built on your Jetson to another Jetson with a different JetPack version. (a) It always works (b) It may fail or give wrong results, so build it again there (c) It becomes faster (d) The file is converted automatically

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `ModuleNotFoundError: No module named 'tensorrt'` | The environment was not made with `--system-site-packages`, or TensorRT is not installed | Call the instructor. Do not run `pip install tensorrt`. |
| Export stops with `export running on CPU but must be on GPU` | PyTorch cannot see the GPU | Run the `check_trt.py` cell. If CUDA is False, call the instructor. |
| Export seems stuck for minutes | Normal. TensorRT is testing many ways to run the model. | Wait. Do not press stop. |
| The build cell says a classmate is building | Only one build fits in the memory at a time | Wait. Your build starts by itself. |
| Export fails with an out-of-memory error | Other notebooks use the shared 8 GB memory | Choose Kernel, Shut Down All Kernels, reopen only this notebook, and try again. |
| Export fails while downloading a package | The export tools (`onnx`, `onnxslim`) are missing and there is no internet | Check the cable and run the Lab 0 setup again. |
| Lines such as `NvMapMemAllocInternalTagged ... error 12` appear | The shared memory is nearly full | The export can still finish. If it stops, shut down all kernels and run it again. |
| `quantize=16 did not work` | Older Ultralytics version | Normal. The program tries `half=True` by itself. |
| `file not found, skipped` in the benchmark | That engine was not built | Run the `ls` cell and repeat Part B. |
| Engine loads but gives errors or no detections | The engine was built on a different JetPack or device | Build it again on this Jetson. |
| FPS in the live view is the same for both models | The camera or the browser limits the FPS | Normal. Use the benchmark (Part C) for the real speed. |
| `Cannot open the camera` | No camera was chosen, or another program holds it | Ask your teacher to choose a camera for you. Press Stop in other live views first. |

## 7. Further Reading
In Lab 16 you use the FP16 engine from this lab to build a complete application.
- Ultralytics: TensorRT Export: https://docs.ultralytics.com/integrations/tensorrt/
- Ultralytics: Quick Start Guide for NVIDIA Jetson: https://docs.ultralytics.com/guides/nvidia-jetson/
- Ultralytics: Export Mode: https://docs.ultralytics.com/modes/export/
- NVIDIA TensorRT Documentation: https://docs.nvidia.com/deeplearning/tensorrt/latest/index.html
- NVIDIA Jetson AI Lab: https://www.jetson-ai-lab.com